# Addint cost models together and obtaining financial metrics

Let's create two cost models with their own costs

In [1]:
import numpy as np
from costmodels import BatteryCostModel
from costmodels import PVCostModel
from costmodels.units import Quant
from costmodels.helpers import calc_cashflows, calc_npv, calc_irr

pv_cm = PVCostModel()
pv_cm_res = pv_cm.run(solar_capacity=Quant(150, "MW"))

bcm = BatteryCostModel()
battery_lifetime = 25
state_of_health = np.hstack(
    [
        -1.7e-6 * np.arange(1.8e5) + 1,
        -2.5e-6 * np.arange(battery_lifetime * 365 * 24 - 1.8e5) + 1,
    ]
).ravel()
bcm_res = bcm.run(
    battery_power=Quant(27, "MW"),
    battery_energy=Quant(108, "MWh"),
    state_of_health=state_of_health,
    battery_energy_onm_cost=1e3,
)

In [2]:
from costmodels.base import OPEX_KEY, CAPEX_KEY

(OPEX_KEY, CAPEX_KEY)

('opex', 'capex')

In [ ]:
pv_cm_res[OPEX_KEY], pv_cm_res["opex"], pv_cm._opex  # all of these are equivalent

(<Quantity(1012500.0, 'EUR')>,
 <Quantity(1012500.0, 'EUR')>,
 <Quantity(1012500.0, 'EUR')>)

Sum up two models, obtaining a single cashflow array for joint lifespan of both projects

In [26]:
np.random.seed(42)
cashflows = calc_cashflows(
    capexs=[x[CAPEX_KEY] for x in [pv_cm_res, bcm_res]],
    opexs=[x[OPEX_KEY] for x in [pv_cm_res, bcm_res]],
    lifetimes=[20, battery_lifetime],
    t0s=[-1, 1],
    prod=Quant(np.ones(12) / 2, "GWh"),
    eprice=Quant(np.random.rand(12), "EUR/kWh"),
)
cashflows

Magnitude,[-48166569.147007726 2083430.8529922722 -5091465.573153908 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 5071361.705984545 2987930.852992273 2987930.852992273 2987930.852992273 2987930.852992273 2987930.852992273 2987930.852992273 2987930.852992273]
Units,EUR


We can obtain multiple finantial metrics from the cashflows

In [23]:
npv = calc_npv(
    discount=Quant(5, "%"),
    cashflows=cashflows,
)
npv.to("MEUR")

<Quantity(7.90078852, 'megaEUR')>

In [24]:
irr = calc_irr(
    cashflows=cashflows,
)
irr

<Quantity(6.39355908, 'percent')>